# Theory 14 — structural observability versus instance reliability

**Formal source:** [`../14_structural_observability_and_instance_reliability.md`](../14_structural_observability_and_instance_reliability.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
gramian_value, threshold, temperature = 2.0, 1.0, 0.2
structural_weight = 1 / (1 + math.exp(-(gramian_value - threshold) / temperature))
assert structural_weight > 0.99
reliability = np.array([1.0, 0.5, 0.1])
posterior_variance = 1 / (1 + 4 * reliability)
assert np.all(np.diff(posterior_variance) > 0)
print({"structural_role_weight": structural_weight, "reliabilities": reliability.tolist(), "posterior_variances": posterior_variance.tolist()})

In [ ]:
print('THEORY_DEMO_PASS::14_structural_observability_and_instance_reliability')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')